# Module 3.2 — Inheritance & Polymorphism

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

Inheritance lets one class build on another, reusing and extending its behaviour. Polymorphism lets different classes respond to the same call in their own way. This module covers single and multiple inheritance, the method resolution order, `super()`, abstract base classes, and duck typing.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Override methods and observe which version runs.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Create a subclass that inherits and extends a base class.
2. Override methods and call the parent version with `super()`.
3. Explain the method resolution order and use multiple inheritance carefully.
4. Define abstract base classes that require certain methods.
5. Use polymorphism and duck typing to write flexible code.

**Prerequisites:** Tracks 1 and 2, and Module 3.1.

---

## Part 1 · Single Inheritance

A subclass is written `class Child(Parent):`. It automatically gains the parent's attributes and methods, and may add its own or replace existing ones. This models an "is a" relationship: a `Dog` is an `Animal`.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"{self.name} is an animal."

    def speak(self):
        return "(some sound)"

class Dog(Animal):                # Dog inherits everything from Animal
    def speak(self):              # override: provide a Dog-specific version
        return "Woof"

d = Dog("Rex")
print(d.describe())               # inherited from Animal, unchanged
print(d.name, "says", d.speak())  # the overridden method runs
print("a Dog is an Animal:", isinstance(d, Animal))

## Part 2 · Extending with `super()`

When a subclass overrides a method but still wants the parent's behaviour, it calls `super()`. This is most common in `__init__`, where the child adds its own data on top of what the parent sets up. Using `super()` rather than naming the parent directly keeps code correct when hierarchies change.

In [ ]:
class Vehicle:
    def __init__(self, brand):
        self.brand = brand

    def info(self):
        return f"{self.brand} vehicle"

class Car(Vehicle):
    def __init__(self, brand, doors):
        super().__init__(brand)      # run the parent initialiser first
        self.doors = doors           # then add this class's own data

    def info(self):
        base = super().info()        # reuse the parent's result
        return f"{base} with {self.doors} doors"

c = Car("Toyota", 4)
print(c.info())
print("brand set by parent:", c.brand)

## Part 3 · Multiple Inheritance and the Method Resolution Order

A class may inherit from several parents. Python decides which method to use through the **method resolution order** (MRO): a single, predictable ordering of all ancestors. You can view it with `ClassName.__mro__` or `mro()`. `super()` follows this order, which is what makes cooperative multiple inheritance work.

A common, disciplined use of multiple inheritance is the **mixin**: a small class that adds one focused capability.

In [ ]:
class Serializable:
    def to_dict(self):
        return self.__dict__

class Comparable:
    def __eq__(self, other):
        return self.__dict__ == other.__dict__

class Record(Serializable, Comparable):    # combine two mixins
    def __init__(self, name, value):
        self.name = name
        self.value = value

r = Record("temp", 20)
print("to_dict from mixin:", r.to_dict())
print("equality from mixin:", r == Record("temp", 20))

# The MRO lists the order Python searches for methods.
print("MRO:", [cls.__name__ for cls in Record.__mro__])

In [ ]:
# A classic illustration of why MRO matters: the diamond shape.
class A:
    def who(self):
        return "A"

class B(A):
    def who(self):
        return "B -> " + super().who()

class C(A):
    def who(self):
        return "C -> " + super().who()

class D(B, C):
    def who(self):
        return "D -> " + super().who()

# super() walks the MRO, so each class is visited exactly once, in order.
print(D().who())                       # D -> B -> C -> A
print([cls.__name__ for cls in D.__mro__])

## Part 4 · Abstract Base Classes

Sometimes a base class should not be instantiated directly and should require subclasses to provide certain methods. The `abc` module supports this: inherit from `abc.ABC` and mark required methods with `@abstractmethod`. Any subclass that fails to implement them cannot be instantiated, which catches mistakes early.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        """Subclasses must implement this."""

    def describe(self):                  # concrete methods are still allowed
        return f"A shape with area {self.area():.2f}"

class Square(Shape):
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

s = Square(3)
print(s.describe())

# Attempting to instantiate the abstract base, or a subclass missing area(), fails:
try:
    Shape()
except TypeError as e:
    print("cannot instantiate Shape:", e)

## Part 5 · Polymorphism and Duck Typing

**Polymorphism** means that code written against a common interface works with any object that provides it. **Duck typing** is Python's informal version: if an object has the method you need, you can use it, regardless of its class. The saying is, "if it walks like a duck and quacks like a duck, it is a duck."

This lets you write functions that accept any suitable object without checking its exact type.

In [ ]:
class Cat:
    def speak(self):
        return "Meow"

class Duck:
    def speak(self):
        return "Quack"

class Robot:
    def speak(self):
        return "Beep"

def make_it_speak(thing):
    # No type check: we only require that 'thing' has a speak() method.
    return thing.speak()

for obj in [Cat(), Duck(), Robot()]:
    print(type(obj).__name__, "->", make_it_speak(obj))

# Polymorphism in action: one loop, different behaviours, no conditionals on type.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A basic subclass (Easy)

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        return "..."

class Cow(Animal):
    def speak(self):
        return "Moo"

c = Cow("Daisy")
print(c.name, "says", c.speak())
# Experiment: add a Sheep subclass that returns "Baa".

### Example 2 — Extending the initialiser with super() (Easy)

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

class Employee(Person):
    def __init__(self, name, salary):
        super().__init__(name)       # set name via the parent
        self.salary = salary

e = Employee("Asha", 50000)
print(e.name, "earns", e.salary)

### Example 3 — Overriding and reusing parent behaviour (Medium)

In [ ]:
class Logger:
    def log(self, message):
        return f"[LOG] {message}"

class TimestampLogger(Logger):
    def log(self, message):
        base = super().log(message)       # reuse the parent format
        return f"2025-05-30 {base}"       # fixed timestamp for reproducibility

print(Logger().log("started"))
print(TimestampLogger().log("started"))

### Example 4 — An abstract base with several implementations (Medium)

In [ ]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        ...

class Card(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} by card"

class Cash(PaymentMethod):
    def pay(self, amount):
        return f"Paid {amount} in cash"

def checkout(method, amount):
    return method.pay(amount)             # works with any PaymentMethod

print(checkout(Card(), 100))
print(checkout(Cash(), 100))

### Example 5 — Cooperative multiple inheritance (Difficult)

In [ ]:
class Engine:
    def __init__(self, **kwargs):
        self.power = kwargs.pop("power", 100)
        super().__init__(**kwargs)        # pass remaining arguments along the MRO

class Radio:
    def __init__(self, **kwargs):
        self.station = kwargs.pop("station", "FM1")
        super().__init__(**kwargs)

class Car(Engine, Radio):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)        # triggers the cooperative chain

    def __repr__(self):
        return f"Car(power={self.power}, station={self.station!r})"

# Each parent takes the arguments it needs and forwards the rest.
print(Car(power=150, station="Jazz"))
print("MRO:", [c.__name__ for c in Car.__mro__])

### Example 6 — Polymorphic rendering with a shared interface (Difficult)

In [ ]:
from abc import ABC, abstractmethod

class Renderer(ABC):
    @abstractmethod
    def render(self, text):
        ...

class HTMLRenderer(Renderer):
    def render(self, text):
        return f"<p>{text}</p>"

class MarkdownRenderer(Renderer):
    def render(self, text):
        return f"{text}\n"

class UpperRenderer(Renderer):
    def render(self, text):
        return text.upper()

def render_all(renderers, text):
    # One function handles every renderer through the common interface.
    return [r.render(text) for r in renderers]

for line in render_all([HTMLRenderer(), MarkdownRenderer(), UpperRenderer()], "hello"):
    print(repr(line))

---

## Exercises

**Exercise 1 (Easy).** Define a base class `Shape` with a method `name()` returning `"shape"`, and a subclass `Triangle` that overrides `name()` to return `"triangle"`. Demonstrate both.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Define `Account` with `__init__(self, owner)`, and a subclass `SavingsAccount` whose `__init__` also takes `rate` and calls `super().__init__`. Print both attributes.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Define a base class `Greeter` with a method `greet()` returning `"Hello"`, and a subclass `PoliteGreeter` whose `greet()` reuses the parent result and appends `", nice to meet you"`. Demonstrate it.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Using `abc`, define an abstract class `Notifier` with an abstract method `send(message)`. Implement two subclasses (`EmailNotifier`, `SMSNotifier`) and a function that sends a message through any notifier.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write a function `total_area(shapes)` that accepts any list of objects, each having an `area()` method (duck typing), and returns the sum of their areas. Test it with at least two unrelated classes that both define `area()`.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
class Shape:
    def name(self):
        return "shape"

class Triangle(Shape):
    def name(self):
        return "triangle"

print(Shape().name())
print(Triangle().name())

**Solution 2**

In [ ]:
class Account:
    def __init__(self, owner):
        self.owner = owner

class SavingsAccount(Account):
    def __init__(self, owner, rate):
        super().__init__(owner)
        self.rate = rate

s = SavingsAccount("Asha", 0.04)
print(s.owner, s.rate)

**Solution 3**

In [ ]:
class Greeter:
    def greet(self):
        return "Hello"

class PoliteGreeter(Greeter):
    def greet(self):
        return super().greet() + ", nice to meet you"

print(PoliteGreeter().greet())

**Solution 4**

In [ ]:
from abc import ABC, abstractmethod

class Notifier(ABC):
    @abstractmethod
    def send(self, message):
        ...

class EmailNotifier(Notifier):
    def send(self, message):
        return f"Email: {message}"

class SMSNotifier(Notifier):
    def send(self, message):
        return f"SMS: {message}"

def notify(notifier, message):
    return notifier.send(message)

print(notify(EmailNotifier(), "hi"))
print(notify(SMSNotifier(), "hi"))

**Solution 5**

In [ ]:
class Square:
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

class Disk:
    def __init__(self, radius):
        self.radius = radius
    def area(self):
        return 3.14159 * self.radius ** 2

def total_area(shapes):
    return sum(shape.area() for shape in shapes)

print(round(total_area([Square(2), Disk(1), Square(3)]), 2))

---

## Summary and Key Points

- A subclass `class Child(Parent):` inherits the parent's attributes and methods and may add to or override them, modelling an "is a" relationship.
- `super()` calls the parent version of a method, most often in `__init__`, and keeps code correct as hierarchies evolve.
- Multiple inheritance follows the method resolution order (`__mro__`), which `super()` respects; mixins are a disciplined, focused use of it.
- Abstract base classes (`abc.ABC` with `@abstractmethod`) cannot be instantiated and force subclasses to implement required methods.
- Polymorphism and duck typing let one piece of code work with any object that provides the expected interface, without checking exact types.

### Next module: 3.3 — Magic Methods, Protocols & Context Managers

The next module covers the special methods that integrate your objects with Python's syntax: representation, equality and hashing, container behaviour, and the context-manager protocol with `contextlib`.